In [1]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

In [2]:
if IN_COLAB:
  # Install dependencies
  ! pip install --upgrade pip
  ! pip install czitools
  ! pip install ipyfilechooser
  ! pip install matplotlib
  ! pip install pandas

In [3]:
from czitools.utils import planetable
from ipyfilechooser import FileChooser
from IPython.display import display
import ipywidgets as widgets
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import os
import requests
import glob

In [4]:
def scatterplot_mpl(planetable,
                    s=0, t=0, z=0, c=0,
                    msz2d=35,
                    normz=True,
                    fig1savename='zsurface2d.png',
                    fig2savename='zsurface3d.png',
                    show3d=False,
                    msz3d=20):

    # extract XYZ positions
    try:
        xpos = planetable['X[micron]']
        ypos = planetable['Y[micron]']
        zpos = planetable['Z[micron]']
    except KeyError as e:
        xpos = planetable['X [micron]']
        ypos = planetable['Y [micron]']
        zpos = planetable['Z [micron]']

    # normalize z-data by subtracting the minimum value
    if normz:
        zpos = zpos - zpos.min()

    # create a name for the figure
    figtitle = 'XYZ-Positions:  S=' + str(s) + ' T=' + str(t) + ' Z=' + str(z) + ' CH=' + str(c)

    # try to find a "good" aspect ratio for the figures
    dx = xpos.max() - xpos.min()
    dy = ypos.max() - ypos.min()
    fsy = 8
    fsx = int(np.ceil(fsy * dx / dy))

    # create figure
    fig1, ax1 = plt.subplots(1, 1, figsize=(fsx + 1, fsy))

    # invert the Y-axis --> O,O = Top-Left
    ax1.invert_yaxis()

    # configure the axis
    ax1.set_title(figtitle)
    ax1.set_xlabel('Stage X-Axis [micron]', fontsize=12, fontweight='normal')
    ax1.set_ylabel('Stage Y-Axis [micron]', fontsize=12, fontweight='normal')
    ax1.grid(True)
    ax1.set_aspect('equal', 'box')

    # plot data and label the colorbar
    sc1 = ax1.scatter(xpos, ypos,
                      marker='s',
                      c=zpos,
                      s=msz2d,
                      facecolor=cm.coolwarm,
                      edgecolor='black')

    # add the colorbar on the right-hand side
    cb1 = plt.colorbar(sc1,
                       fraction=0.046,
                       shrink=0.8,
                       pad=0.04)

    # add a label
    if normz:
        cb1.set_label('Z-Offset [micron]',
                      labelpad=20,
                      fontsize=12,
                      fontweight='normal')
    if not normz:
        cb1.set_label('Z-Position [micron]',
                      labelpad=20,
                      fontsize=12,
                      fontweight='normal')

    # save figure as PNG
    fig1.savefig(fig1savename, dpi=100)
    print('Saved: ', fig1savename)

    # 3D plot of surface
    fig2 = plt.figure(figsize=(fsx + 1, fsy))
    ax2 = fig2.add_subplot(111, projection='3d')

    # invert the Y-axis --> O,O = Top-Left
    ax2.invert_yaxis()

    # define the labels
    ax2.set_xlabel('Stage X-Axis [micron]',
                   fontsize=12,
                   fontweight='normal')
    ax2.set_ylabel('Stage Y-Axis [micron]',
                   fontsize=12,
                   fontweight='normal')
    ax2.set_title(figtitle)

    # plot data and label the colorbar
    sc2 = ax2.scatter(xpos, ypos, zpos,
                      marker='.',
                      s=msz3d,
                      c=zpos,
                      facecolor=cm.coolwarm,
                      depthshade=False)

    # add colorbar to the 3d plot
    cb2 = plt.colorbar(sc2, shrink=0.8)
    # add a label
    if normz:
        cb2.set_label('Z-Offset [micron]',
                      labelpad=20,
                      fontsize=12,
                      fontweight='normal')
    if not normz:
        cb2.set_label('Z-Position [micron]',
                      labelpad=20,
                      fontsize=12,
                      fontweight='normal')

    # save figure as PNG
    fig2.savefig(fig2savename, dpi=100)
    print('Saved: ', fig2savename)

    return fig1, fig2


In [5]:
# try to find the folder with data and download otherwise from GitHub.

# Folder containing the input data
INPUT_FOLDER = 'data/'

# Path to the data on GitHub
GITHUB_IMAGES_PATH = "https://raw.githubusercontent.com/sebi06/ZEN_Python_Workshop/main/notebooks/data.zip"

# Download data
if not (os.path.isdir(INPUT_FOLDER)):
    compressed_data = './data.zip'
    if not os.path.isfile(compressed_data):
        import io
        response = requests.get(GITHUB_IMAGES_PATH, stream=True)
        compressed_data = io.BytesIO(response.content)

    import zipfile
    with zipfile.ZipFile(compressed_data, 'r') as zip_accessor:
        zip_accessor.extractall('./')

In [6]:
if not IN_COLAB:
    # choose local file
    fc = FileChooser()
    fc.default_path = INPUT_FOLDER
    fc.filter_pattern = '*.czi'
    display(fc)

elif IN_COLAB:
    # lislt files inside the folder om gdrive
    czifiles = glob.glob(os.path.join(INPUT_FOLDER, "*.czi"))
    wd = widgets.Select(
        options=czifiles,
        description='CZI Files:',
        layout={'width': 'max-content'}
    )
    display(wd)

FileChooser(path='F:\GitHub\ZEN_Python_Workshop\notebooks\data', filename='', title='', show_hidden=False, sel…

In [11]:
if not IN_COLAB:
    filepath = fc.selected
elif IN_COLAB:
    filepath = wd.value

print(f"Selected File: {filepath}")

Selected File: F:\GitHub\ZEN_Python_Workshop\notebooks\data\testwell96_small.czi


In [16]:
# read the data from CZI file
planes, savepath = planetable.get_planetable(filepath,
                                             norm_time=True,
                                             save_table=True,
                                             planes={"channel": 0}
                                            )

planes

  0% |                                                 | ETA:  --:--:-- 0 of 192
100% |###############################################| Time:  0:00:00 192 of 192


2026-05-18 08:47:09,384 - czitools - INFO - Planetable saved successfully at: F:\GitHub\ZEN_Python_Workshop\notebooks\data\testwell96_small_planetable.csv


,Subblock,S,M,T,C,Z,X[micron],Y[micron],Z[micron],Time[s],xstart,ystart,width,height
0,0,0,0,0,0,0,4261.284,0.000,0.0,0.0,42,0,61,61
1,1,0,1,0,0,0,13293.666,8372.441,0.0,0.0,4981,6,61,61
2,2,0,2,0,0,0,22326.052,8383.476,0.0,0.0,9921,12,61,61
3,3,0,3,0,0,0,31358.388,8394.511,0.0,0.0,14860,18,61,61
4,4,0,4,0,0,0,40390.804,8405.546,0.0,0.0,19800,24,61,61
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,91,0,91,0,0,0,40313.719,71483.451,0.0,0.0,19758,34519,61,61
92,92,0,92,0,0,0,31281.332,71472.416,0.0,0.0,14818,34513,61,61
93,93,0,93,0,0,0,22248.964,71461.381,0.0,0.0,9879,34507,61,61
94,94,0,94,0,0,0,13216.612,71450.346,0.0,0.0,4939,34501,61,61


In [ ]:
# define name for figure to be saved
fig1savename = str(Path(INPUT_FOLDER) / (Path(filepath).stem + "_XYZ-Pos.png"))
fig2savename = str(Path(INPUT_FOLDER) / (Path(filepath).stem + "_XYZ-Pos3D.png"))

# display the XYZ positions using matplotlib
fig1, fig2 = scatterplot_mpl(planes,
                             s=0, t=0, z=0, c=0,
                             msz2d=100,
                             normz=True,
                             fig1savename=fig1savename,
                             fig2savename=fig2savename,
                             msz3d=10
                            )

if not IN_COLAB:
    pyo.iplot(fig1)
    pyo.iplot(fig2)

if IN_COLAB:
    fig1.show(renderer="colab")
    fig2.show(renderer="colab")



Saved:  ..\..\data\WP96_4Pos_B4-10_DAPI_XYZ-Pos.html
Saved:  ..\..\data\WP96_4Pos_B4-10_DAPI_XYZ-Pos3D.html


In [15]:
# write the planetable to a csv
csvfile = planetable.save_planetable(planes, filepath,
                                separator="\t",
                                index=False)

print('Write to CSV File : ', csvfile)

Write to CSV File :  F:\GitHub\czitools\data\WP96_4Pos_B4-10_DAPI_planetable.csv
